# Session 2: resistor networks and a design to a limit
**A. Real-world problem.** You have a 12 V supply and small 0.25 W resistors. You want about
0.1 A to flow. One resistor of the right size would overheat. Can several resistors share the
load safely?

**Plan.** (1) series and parallel rules, (2) current and power in every resistor,
(3) search the standard E12 values for the smallest safe design.

In [ ]:
import itertools
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

E12 = (1.0, 1.2, 1.5, 1.8, 2.2, 2.7, 3.3, 3.9, 4.7, 5.6, 6.8, 8.2)   # standard values per decade

## Step 1: equivalent resistance
Series (one path): the same current flows through all, R = R1 + R2 + ...
Parallel (several paths): the same voltage is across all, 1/R = 1/R1 + 1/R2 + ...
Check by hand: 6 ohm in series with 12 ohm is 18 ohm; 6 ohm parallel with 12 ohm is
6*12/(6+12) = 4 ohm.

In [ ]:
def series(rs):
    return float(sum(rs))


def parallel(rs):
    return float(1.0 / sum(1.0 / r for r in rs))

## Step 2: a grid of identical resistors
`n_s` resistors in series make one branch; `n_p` such branches sit in parallel. Each
resistor has the value `r`. Total R = n_s * r / n_p. Total current I = V / R, and each
branch carries I / n_p, so each resistor dissipates (I / n_p)^2 * r.

In [ ]:
def grid_resistance(r, n_s, n_p):
    return parallel([series([r] * n_s)] * n_p)


def grid_per_resistor_power(volts, r, n_s, n_p):
    """Power (W) in each resistor of the grid."""
    i_total = volts / grid_resistance(r, n_s, n_p)
    return (i_total / n_p) ** 2 * r

## Step 3: design search
Find all grids of E12 values (decades from 1 ohm to
8.2 kilohm) with at most `max_n` resistors, whose current is within `tol` of the target and
where every resistor stays under its power rating. Sort by the number of resistors.

In [ ]:
def design(volts, i_target, tol, rating, max_n=8):
    """List of dicts (n, n_s, n_p, r, current, p_each) meeting the limits, fewest parts first."""
    found = []
    values = [v * 10 ** d for d in range(0, 4) for v in E12]
    for n_s, n_p in itertools.product(range(1, max_n + 1), repeat=2):
        if n_s * n_p > max_n:
            continue
        for r in values:
            i = volts / grid_resistance(r, n_s, n_p)
            p_each = grid_per_resistor_power(volts, r, n_s, n_p)
            if abs(i - i_target) <= tol * i_target and p_each <= rating:
                found.append({"n": n_s * n_p, "n_s": n_s, "n_p": n_p, "r": r, "current": i, "p_each": p_each})
    found.sort(key=lambda d: (d["n"], abs(d["current"] - i_target)))
    return found


if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    print("series(6, 12) =", series([6, 12]), "ohm; parallel(6, 12) =", parallel([6, 12]), "ohm")
    print("one 120 ohm resistor on 12 V dissipates", 12 ** 2 / 120, "W (rating 0.25 W!)")
    best = design(12.0, 0.100, 0.05, 0.25)
    for d in best[:5]:
        print(f"{d['n']} parts: {d['n_s']} in series x {d['n_p']} branches of {d['r']:g} ohm -> "
              f"I = {1000 * d['current']:.1f} mA, each part {d['p_each']:.3f} W")

    # P vs. number of series parts for total R fixed near 120 ohm
    fig, ax = plt.subplots(figsize=(6, 4))
    ns = np.arange(1, 9)
    p_each = [(12.0 / 120.0) ** 2 * (120.0 / n) for n in ns]
    ax.bar(ns, p_each, color="#3b6fb6")
    ax.axhline(0.25, color="crimson", ls="--", label="rating 0.25 W")
    ax.set_xlabel("identical resistors in series, total 120 ohm")
    ax.set_ylabel("power in each resistor (watts)")
    ax.set_title("Sharing the load: 12 V, 0.1 A")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "share_the_load.png", dpi=120)

    # design space: total R vs V -> current and per-part limit
    v = np.linspace(1, 24, 120)
    r_tot = np.linspace(5, 300, 120)
    VV, RR = np.meshgrid(v, r_tot)
    II = VV / RR
    PP = VV ** 2 / RR
    fig2, ax2 = plt.subplots(figsize=(6, 4.5))
    m = ax2.pcolormesh(VV, RR, PP, shading="auto", cmap="viridis")
    fig2.colorbar(m, ax=ax2, label="power P (watts)")
    ax2.contour(VV, RR, II, levels=[0.1], colors="white")
    ax2.contour(VV, RR, PP, levels=[0.25], colors="red")
    ax2.set_xlabel("supply voltage V (volts)")
    ax2.set_ylabel("resistance R (ohms)")
    ax2.set_title("Power map; white: I = 0.1 A, red: P = 0.25 W")
    fig2.tight_layout()
    fig2.savefig(OUT / "design_map.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Use `series` and `parallel` to check the Section F answer B: 6 ohm in series with two 12 ohm in parallel.
2. **Stretch.** Call `design(12.0, 0.050, 0.05, 0.25)` and print the best three designs; confirm the first one by hand.
3. **Explain.** Explain why the best grid for 0.1 A at 12 V needs six parts, not five, even though five ideal 24 ohm resistors would do (hint: E12).

## Assessment
- **A. Conceptual.** Why does adding a resistor in parallel make the total resistance smaller, even though you added more material?
- **B. Numerical.** A 12 V supply feeds a 6 ohm resistor in series with two 12 ohm resistors in parallel. Find the total current, the power in the 6 ohm resistor and the power in each 12 ohm resistor.
- **C. Graph and simulation interpretation.** In the power graph for a single 120 ohm resistor, at what supply voltage does the curve cross the 0.25 W rating? Where does it cross for the 2 x 3 grid of 180 ohm resistors?
- **D. Design and optimisation challenge.** Design: with a 12 V supply and 0.25 W resistors, get 50 mA within 5 percent using the fewest E12 resistors. State the network and its checks.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("hand calculation: 6 ohm + 12 ohm in series is 18 ohm", abs(L.series([6, 12]) - 18.0) < 1e-12))
    checks.append(("hand calculation: 6 ohm || 12 ohm is 4 ohm", abs(L.parallel([6, 12]) - 4.0) < 1e-12))
    checks.append(("two equal resistors in parallel give half", abs(L.parallel([50, 50]) - 25.0) < 1e-12))
    checks.append(("parallel is smaller than the smallest branch", L.parallel([3, 8, 20]) < 3))
    checks.append(("worked example: 2 x 3 grid of 180 ohm is 120 ohm and 0.100 A on 12 V",
                   abs(L.grid_resistance(180, 2, 3) - 120.0) < 1e-9 and abs(12 / L.grid_resistance(180, 2, 3) - 0.1) < 1e-9))
    checks.append(("worked example: each of the six parts dissipates 0.200 W",
                   abs(L.grid_per_resistor_power(12, 180, 2, 3) - 0.2) < 1e-9))
    # total power equals V*I
    total = 6 * L.grid_per_resistor_power(12, 180, 2, 3)
    checks.append(("sum of part powers equals V*I = 1.2 W", abs(total - 1.2) < 1e-9))
    d = L.design(12.0, 0.100, 0.05, 0.25)
    checks.append(("smallest safe design needs 6 parts", d[0]["n"] == 6))
    checks.append(("no design with 5 or fewer parts exists", all(x["n"] >= 6 for x in d)))
    checks.append(("every design respects the rating and the current tolerance",
                   all(x["p_each"] <= 0.25 and abs(x["current"] - 0.1) <= 0.005 + 1e-12 for x in d)))
    d50 = L.design(12.0, 0.050, 0.05, 0.25)[0]
    checks.append(("section F design: three 82 ohm in series give 48.8 mA at 0.195 W each",
                   d50["n"] == 3 and d50["r"] == 82 and abs(d50["current"] - 0.04878) < 1e-4 and abs(d50["p_each"] - 0.1951) < 1e-3))
    checks.append(("a single 120 ohm resistor fails the 0.25 W rating", 12 ** 2 / 120 > 0.25))
    checks.append(("impossible limit gives no design", L.design(12.0, 0.100, 0.05, 0.001, max_n=4) == []))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
